# NB16 — Clade-Stratified PGLS

**Type:** Exploratory  
**Status:** Reconstructed from cached data (original ran via Spark on kescience_mgnify).  

**Database note:** The original clade-stratified analysis used  Spark queries
with within-phylum predictor standardisation (same pipeline as NB03). This reconstruction
uses the -derived predictor from  (same DB as P1).
The t-statistics and significance results agree exactly; the β values differ in scale because the two
databases have different KO-density distributions. The archived values in
 preserve the original (kescience_mgnify-derived) results
cited in REPORT.md. The reconstructed notebook verifies statistical significance and direction.

Within-phylum PGLS to test whether the primary metal-gene–niche breadth signal (β = −0.021)
is consistent across the four best-represented phyla: Proteobacteria, Firmicutes,
Actinobacteria, Bacteroidetes.

Also computes Cochran Q heterogeneity test across the four phyla.

## Inputs
-  — PGLS input (genus, ko_per_mb_primary, niche breadth, phylum)
-  — pruned GTDB bacterial genus tree

## Outputs (reconstruction; original values in archived CSV)
-  — original kescience_mgnify values (archived)
-  — forest plot


In [1]:
import sys
from pathlib import Path

_project_root = Path().resolve().parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

from scripts.pgls_utils import run_pgls

DATA = Path('../data')
FIGS = Path('../figures')
FIGS.mkdir(exist_ok=True)
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'

bac_df = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
print(f'Loaded PGLS input: {len(bac_df)} genera')
print('Columns:', bac_df.columns.tolist())
print('\nPhylum distribution:')
print(bac_df['phylum'].value_counts().head(10))

Loaded PGLS input: 1574 genera
Columns: ['genus_lower', 'ko_per_mb_primary', 'mean_genome_mb', 'mean_levins_B_std', 'phylum', 'kingdom', 'predictor_z', 'genome_mb_z']

Phylum distribution:
phylum
Proteobacteria     677
Firmicutes         334
Actinobacteria     204
Bacteroidetes      183
Cyanobacteria       27
Planctomycetes      17
Chloroflexi         16
Verrucomicrobia     14
Spirochaetes        11
Synergistetes       11
Name: count, dtype: int64


In [2]:
# Run within-phylum PGLS for four best-represented phyla
PHYLA = ['Proteobacteria', 'Firmicutes', 'Actinobacteria', 'Bacteroidetes']

results = []
for phylum in PHYLA:
    sub = bac_df[bac_df['phylum'] == phylum].copy()
    sub = sub.dropna(subset=['predictor_z', 'mean_levins_B_std'])
    print(f'\n{phylum}: n = {len(sub)}')
    if len(sub) < 30:
        print(f'  SKIPPED — fewer than 30 genera')
        continue
    try:
        r = run_pgls(
            sub,
            str(TREE_BAC),
            response='mean_levins_B_std',
            predictors=['predictor_z'],
            taxon_col='genus_lower',
            label=phylum,
            min_n=30
        )
        r['phylum'] = phylum
        results.append(r)
        beta = r.get('beta', float('nan'))
        se   = r.get('se', r.get('SE', float('nan')))
        pval = r.get('p_value', float('nan'))
        lam  = r.get('lambda_est', float('nan'))
        n    = r.get('n', len(sub))
        print(f'  β = {beta:.4f}, SE = {se:.4f}, p = {pval:.4e}, λ = {lam:.4f}, n = {n}')
    except Exception as e:
        print(f'  ERROR: {e}')

print(f'\nCompleted: {len(results)} phyla')


Proteobacteria: n = 677


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  β = -0.0242, SE = 0.0059, p = 4.4112e-05, λ = 0.7775, n = 677

Firmicutes: n = 334


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  β = -0.0137, SE = 0.0071, p = 5.4681e-02, λ = 0.6291, n = 334

Actinobacteria: n = 204


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  β = -0.0297, SE = 0.0091, p = 1.2513e-03, λ = 0.7793, n = 204

Bacteroidetes: n = 183


  β = -0.0126, SE = 0.0148, p = 3.9677e-01, λ = 0.9174, n = 183

Completed: 4 phyla


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


In [3]:
# Cochran's Q heterogeneity test across phyla
betas = np.array([r.get('beta', np.nan) for r in results])
ses   = np.array([r.get('se', r.get('SE', np.nan)) for r in results])
ns    = np.array([r.get('n', np.nan) for r in results])

valid = ~(np.isnan(betas) | np.isnan(ses) | (ses == 0))
betas_v = betas[valid]; ses_v = ses[valid]
weights = 1.0 / ses_v**2
beta_fe = np.sum(weights * betas_v) / np.sum(weights)  # fixed-effect pooled β

Q = np.sum(weights * (betas_v - beta_fe)**2)
df_q = np.sum(valid) - 1
p_q  = 1 - stats.chi2.cdf(Q, df=df_q)
I2   = max(0, (Q - df_q) / Q * 100)

print(f'Fixed-effect pooled β = {beta_fe:.4f}')
print(f"Cochran's Q = {Q:.2f}, df = {df_q}, p = {p_q:.3f}")
print(f'I² = {I2:.1f}%')
print('\nInterpretation: no significant heterogeneity across phyla' if p_q > 0.05 else
      '\nInterpretation: significant heterogeneity detected')

Fixed-effect pooled β = -0.0212
Cochran's Q = 2.61, df = 3, p = 0.456
I² = 0.0%

Interpretation: no significant heterogeneity across phyla


In [4]:
# Build and save output CSV
rows = []
for r in results:
    beta = r.get('beta', np.nan)
    se   = r.get('se', r.get('SE', np.nan))
    pval = r.get('p_value', np.nan)
    lam  = r.get('lambda_est', np.nan)
    n    = r.get('n', np.nan)
    r2   = r.get('r2', np.nan)
    # n_eff from lambda: n_eff ≈ n * (1 - lambda) + 2 (rough approx)
    n_eff = r.get('n_eff', np.nan)
    ci_lo = beta - 1.96 * se
    ci_hi = beta + 1.96 * se
    rows.append({
        'label': r.get('phylum', r.get('label', '?')),
        'n': n,
        'lambda_est': round(lam, 4),
        'beta': round(beta, 10),
        'SE': round(se, 10),
        'p': pval,
        'r2': round(r2, 10) if not np.isnan(r2) else np.nan,
        'ci_lo': round(ci_lo, 10),
        'ci_hi': round(ci_hi, 10),
        'q_bh': np.nan,   # filled below
        'significant': pval < 0.05 if not np.isnan(pval) else False,
        'n_eff': round(n_eff, 1) if not np.isnan(n_eff) else np.nan,
    })

# BH FDR within phyla
from statsmodels.stats.multitest import multipletests
pvals_arr = np.array([r['p'] for r in rows])
valid_mask = ~np.isnan(pvals_arr)
if valid_mask.any():
    _, q_adj, _, _ = multipletests(pvals_arr[valid_mask], method='fdr_bh')
    q_all = np.full(len(rows), np.nan)
    q_all[valid_mask] = q_adj
    for i, row in enumerate(rows):
        row['q_bh'] = q_all[i]

out_df = pd.DataFrame(rows)
out_df.to_csv(DATA / 'clade_stratified_pgls_results.csv', index=False)
print('Saved: data/clade_stratified_pgls_results.csv')
print(out_df[['label','n','lambda_est','beta','SE','p','q_bh','significant','n_eff']].to_string(index=False))

Saved: data/clade_stratified_pgls_results.csv
         label   n  lambda_est      beta       SE        p     q_bh  significant  n_eff
Proteobacteria 677      0.7775 -0.024152 0.005874 0.000044 0.000176         True    NaN
    Firmicutes 334      0.6291 -0.013654 0.007081 0.054681 0.072908        False    NaN
Actinobacteria 204      0.7793 -0.029712 0.009078 0.001251 0.002503         True    NaN
 Bacteroidetes 183      0.9174 -0.012557 0.014783 0.396773 0.396773        False    NaN


In [5]:
# Forest plot — clade-stratified β ± 95% CI
fig, ax = plt.subplots(figsize=(7, 4))

labels = [r['label'] for r in rows]
betas_p = [r['beta'] for r in rows]
ci_lo_p = [r['ci_lo'] for r in rows]
ci_hi_p = [r['ci_hi'] for r in rows]
pvals_p = [r['p'] for r in rows]
ns_p    = [r['n'] for r in rows]

colors = ['#d62728' if p < 0.05 else '#aec7e8' for p in pvals_p]
y_pos  = list(range(len(labels)))

for i, (y, b, lo, hi, col) in enumerate(zip(y_pos, betas_p, ci_lo_p, ci_hi_p, colors)):
    ax.plot([lo, hi], [y, y], color=col, lw=1.5, zorder=2)
    ax.plot(b, y, 'o', color=col, ms=7, zorder=3)

# Primary result line
ax.axvline(-0.0207, color='#2ca02c', lw=1.2, ls='--', label='Primary β = −0.021 (P1)')
ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.4)

ax.set_yticks(y_pos)
ax.set_yticklabels([f'{l} (n={int(n)})' for l, n in zip(labels, ns_p)], fontsize=9)
ax.set_xlabel('β (ko_per_mb_z → niche breadth)', fontsize=9)
ax.set_title('Clade-stratified PGLS: within-phylum β ± 95% CI', fontsize=10)
ax.legend(fontsize=8, framealpha=0.7)

# Annotation: Q test result
ax.text(0.98, 0.04, f"Cochran's Q={Q:.2f}, p={p_q:.3f}, I²={I2:.0f}%",
        transform=ax.transAxes, ha='right', va='bottom', fontsize=7.5,
        color='#555555')

ax.invert_yaxis()
plt.tight_layout()
fig.savefig(FIGS / 'clade_stratified_forest_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/clade_stratified_forest_plot.png')
print(f"\nCochran's Q={Q:.2f} (df={df_q}, p={p_q:.3f}), I²={I2:.1f}%")

Saved: figures/clade_stratified_forest_plot.png

Cochran's Q=2.61 (df=3, p=0.456), I²=0.0%
